# Построение пайплайна обучения линейной модели


## Генерация данных

In [1]:
import pandas as pd
from sklearn.datasets import make_regression

# Генерация данных
# 200 примеров, 5 признаков, шум с дисперсией 15
# фиксированное random_state для воспроизводимости
X, y = make_regression(n_samples=200,
                       n_features=5,
                       noise=15,
                       random_state=42)

# Преобразуем в DataFrame и Series
X = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
y = pd.Series(y, name='target')

# Добавим колонку с уникальным идентификатором записи
X.insert(0, 'record_id', range(len(X)))

# Объединим признаки и целевую переменную
df = pd.concat([X, y], axis=1)

print(df.head(5))

   record_id  feature_0  feature_1  feature_2  feature_3  feature_4  \
0          0  -0.385314   0.199060  -0.600217   0.462103   0.069802   
1          1   0.130741   1.632411  -1.430141  -1.247783  -0.440044   
2          2  -0.773010   0.224092   0.012592  -0.401220   0.097676   
3          3  -0.576771  -0.050238  -0.238948   0.270457  -0.907564   
4          4  -0.575818   0.614167   0.757508  -0.220970  -0.530501   

       target  
0  -19.876024  
1 -121.994064  
2   19.221414  
3  -61.045683  
4   27.922024  


## Разбиение на обучающую, валидационную и тестовую выборки

Задание 1

Вы подготовили набор данных для задачи регрессии. Теперь необходимо:
- Удалить колонку record_id — это технический идентификатор, который не несёт полезной информации для модели.
- Разделить датафрейм df на обучающую, валидационную и тестовую выборки в соотношении 60/20/20, отдельно сохранив признаки ( X_train, X_val, X_test ) и целевую переменную ( y_train, y_val, y_test ). Зафиксируйте случайность — укажите random_state=42, а также перемешайте данные перед разбиением.
- Проверьте размерности полученных данных.

In [2]:
from sklearn.model_selection import train_test_split

# Удалите колонку с индексом в df
df = df.drop('record_id', axis=1)  # напишите ваш код здесь

# Отделите признаки и целевую переменную (target)
X = df.drop('target', axis=1).values  # напишите ваш код здесь
y = df['target'].values  # напишите ваш код здесь

# Выделите на train_val и test
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True) # напишите ваш код здесь

# Выделите на train и val
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, shuffle=True) # напишите ваш код здесь

# Проверьте размерности полученных переменных
print(X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape)

(120, 5) (40, 5) (40, 5) (120,) (40,) (40,)


## Предобработка данных

 Будем реализовывать масштабирование признаков. Это особенно критично для моделей, чувствительных к масштабу входных данных, таких как линейная регрессия.

Задание 2

Реализуйте функцию scale_data(), которая принимает X_train, X_val и X_test, а также аргумент method, который определяет тип масштабирования (стандартизация: standard, Мин-Макс: minmax ) и возвращает масштабированные версии входных признаков.

Примените функцию для стандартизации ( method='standard' ) признаков, полученных на прошлом шаге ( X_train, X_val и X_test ). Для проверки результата масштабирования выведите средние значения средних и стандартных отклонений для X_train_scaled, X_val_scaled и X_test_scaled, округлённые до двух знаков после запятой.

In [3]:
def scale_data(X_train, X_val, X_test, method='standard'):
	if method == 'standard':
   		mean = X_train.mean() # ваш код здесь
   		std = X_train.std() # ваш код здесь
   		X_train_scaled = (X_train - mean) / std # ваш код здесь
   		X_val_scaled = (X_val - mean) / std # ваш код здесь
   		X_test_scaled = (X_test - mean) / std # ваш код здесь
	elif method == 'minmax':
   		min_val = X_train.min() # ваш код здесь
   		max_val = X_train.max()  # ваш код здесь
   		X_train_scaled = (X_train - min_val) / (max_val - min_val)  # ваш код здесь
   		X_val_scaled = (X_val - min_val) / (max_val - min_val) # ваш код здесь
   		X_test_scaled = (X_test - min_val) / (max_val - min_val) # ваш код здесь
	else:
   		raise ValueError("Неверный метод масштабирования. Используйте 'standard' или 'minmax'.")
	return X_train_scaled, X_val_scaled, X_test_scaled

X_train_scaled, X_val_scaled, X_test_scaled = scale_data(X_train, X_val, X_test) # ваш код здесь

# Проверка: вывод среднего и стандартного отклонения полученных выборок
print("Train mean (avg):", X_train_scaled.mean().mean().round(2))
print("Train std (avg):", X_train_scaled.std().mean().round(2))
print("Val mean (avg):", X_val_scaled.mean().mean().round(2))
print("Val std (avg):", X_val_scaled.std().mean().round(2))
print("Test mean (avg):", X_test_scaled.mean().mean().round(2))
print("Test std (avg):", X_test_scaled.std().mean().round(2))

Train mean (avg): -0.0
Train std (avg): 1.0
Val mean (avg): 0.06
Val std (avg): 0.96
Test mean (avg): -0.08
Test std (avg): 0.97


## Обучение линейной модели

Первым делом обучим обычную линейную регрессию — без регуляризации, и выведем полученные коэффициенты модели, округлённые до двух знаков после запятой.

In [4]:
import numpy as np
# Импорт модели
from sklearn.linear_model import LinearRegression

# Инициализация и обучение
linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train)

# Коэффициенты модели
print(np.round(linreg.coef_, 2))

[ 1.98 12.53 64.79 18.8  70.15]


Задание 3

Теперь самостоятельно обучите Ridge -регрессию — вариант линейной модели с L2-регуляризацией, который помогает бороться с переобучением, особенно если признаки коррелируют между собой. Обучите модель Ridge с параметром alpha = 1.0 на тех же масштабированных признаках и также выведите её коэффициенты, округлённые до двух знаков.

In [5]:
# Импорт модели
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

# Коэффициенты модели
print(np.round(ridge.coef_, 2))


[ 2.05 12.42 64.28 18.71 69.61]


## Получение предсказаний

Следующий шаг — получение предсказаний. На этом этапе вы подаёте в модель признаки, и она выдаёт прогнозы — предполагаемые значения целевой переменной.

Получим предсказания как для обучающей выборки, так и для валидационной и тестовой — это пригодится на следующем этапе, когда будем оценивать качество модели.

In [6]:
# Предсказания для обучающей, валидационной и тестовой выборок
# Линейная регрессия:
y_train_pred_lr = linreg.predict(X_train_scaled)
y_val_pred_lr = linreg.predict(X_val_scaled)
y_test_pred_lr = linreg.predict(X_test_scaled)

# Ridge:
y_train_pred_ridge = ridge.predict(X_train_scaled)
y_val_pred_ridge = ridge.predict(X_val_scaled)
y_test_pred_ridge = ridge.predict(X_test_scaled)

## Расчёт метрик

Нужно понять, насколько хорошо модель справилась с задачей. Для этого рассчитайте метрики качества.

Задание 4

Реализуйте функцию calculate_metrics() для расчёта RMSE, MAPE и R².

Примените её к обучающей, валидационной и тестовой выборкам для расчёта указанных метрик для модели Ridge. Выведите значения метрик на трёх выборках.

In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import numpy as np


def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred)) # напишите ваш код здесь
    mape = mean_absolute_percentage_error(y_true, y_pred) # напишите ваш код здесь
    r2 = r2_score(y_true, y_pred) # напишите ваш код здесь
    return {'RMSE': round(rmse, 2),
            'MAPE': round(mape * 100, 2),
            'R2': round(r2, 3)}

ridge_train_metrics = calculate_metrics(y_train, y_train_pred_ridge ) # напишите ваш код здесь
ridge_val_metrics = calculate_metrics(y_val, y_val_pred_ridge )# напишите ваш код здесь
ridge_test_metrics = calculate_metrics(y_test, y_test_pred_ridge ) # напишите ваш код здесь

print("Метрики Ridge на train:", ridge_train_metrics)
print("Метрики Ridge на val:", ridge_val_metrics)
print("Метрики Ridge на test:", ridge_test_metrics)

Метрики Ridge на train: {'RMSE': np.float64(14.28), 'MAPE': 42.25, 'R2': 0.981}
Метрики Ridge на val: {'RMSE': np.float64(18.02), 'MAPE': 40.96, 'R2': 0.941}
Метрики Ridge на test: {'RMSE': np.float64(14.19), 'MAPE': 42.28, 'R2': 0.977}


Аналогично рассчитаем метрики RMSE, MAPE и R² для модели линейной регрессии:

In [8]:
linreg_train_metrics = calculate_metrics(y_train, y_train_pred_lr)
linreg_val_metrics = calculate_metrics(y_val, y_val_pred_lr)
linreg_test_metrics = calculate_metrics(y_test, y_test_pred_lr)

print("Метрики Linear Regression на train:", linreg_train_metrics)
print("Метрики Linear Regression на val:", linreg_val_metrics)
print("Метрики Linear Regression на test:", linreg_test_metrics)

Метрики Linear Regression на train: {'RMSE': np.float64(14.26), 'MAPE': 42.4, 'R2': 0.981}
Метрики Linear Regression на val: {'RMSE': np.float64(18.19), 'MAPE': 41.33, 'R2': 0.94}
Метрики Linear Regression на test: {'RMSE': np.float64(14.34), 'MAPE': 42.62, 'R2': 0.976}


Делать выводы пока рано. Важно проверить, насколько модель стабильна, — сравнить её поведение на train  и val. 

Идеальный вариант — когда качество на обеих выборках примерно одинаковое.

## Сравнение метрик на train и val

Для наглядности рассчитаем относительные изменения метрик: такая интерпретация изменения метрик между train и val поможет понять, насколько модель переобучилась или недообучилась.

Реализуем функцию для расчёта относительного изменения метрик между выборками. Применим её для Ridge -модели и линейной регрессии.

In [9]:
def compare_metrics(train_metrics, val_metrics):
    for metric in train_metrics:
        change = (val_metrics[metric] - train_metrics[metric]) / train_metrics[metric] * 100
        direction = "изменилась" if metric != 'R2' else "изменился"
        print(f"{metric} на val {direction} по сравнению с {metric} на train на {change:.2f}%")

# Расчёт относительного изменения метрик
print('Ridge:')
compare_metrics(ridge_train_metrics, ridge_val_metrics)
print('Linear Regression:')
compare_metrics(linreg_train_metrics, linreg_val_metrics)

Ridge:
RMSE на val изменилась по сравнению с RMSE на train на 26.19%
MAPE на val изменилась по сравнению с MAPE на train на -3.05%
R2 на val изменился по сравнению с R2 на train на -4.08%
Linear Regression:
RMSE на val изменилась по сравнению с RMSE на train на 27.56%
MAPE на val изменилась по сравнению с MAPE на train на -2.52%
R2 на val изменился по сравнению с R2 на train на -4.18%


Для переобучения расхождения должны быть 50%

Обе модели показывают очень похожую стабильность: рост RMSE умеренный, MAPE и R² немного падают (что допустимо из-за случайного шума в данных). Это говорит о достаточно стабильном поведении моделей на новых данных.

Делаем вывод, что обе модели показывают схожее качество и стабильность, но регуляризация даёт Ridge-регрессии лёгкое преимущество — делает её более устойчивым и потенциально более надёжным вариантом для применения на новых данных.

## Финальная оценка качества


Выведем рассчитанные ранее значения метрик финальной модели (Ridge -регрессии) на тестовой выборке. Поскольку эта выборка не использовалась при обучении, её результаты дают объективное и честное представление о качестве модели.

In [10]:
print("Метрики на test:", ridge_test_metrics)

Метрики на test: {'RMSE': np.float64(14.19), 'MAPE': 42.28, 'R2': 0.977}


Проанализируем полученные значения:

- RMSE = 14,19. Это средняя ошибка модели в тех же единицах, что и целевая переменная. В среднем предсказания отклоняются от истинных значений примерно на 14 единиц. Насколько это много или мало — зависит от масштаба целевой переменной.
- MAPE = 42,29%. В среднем предсказания ошибаются на ~42% относительно истинного значения. Это довольно высокая ошибка в относительных терминах — значит, на малых значениях модель часто ошибается существенно.
- R² = 0,977. Модель объясняет 97,7% дисперсии целевой переменной. Это очень высокий показатель качества, говорящий о том, что модель хорошо улавливает общие закономерности.

В полученных метриках есть некоторое противоречие: высокий R² и одновременно высокий MAPE. Это может означать, что:

- в данных есть малые значения таргета, где относительная ошибка становится огромной (из-за деления на маленькое число), поэтому MAPE становится малоинформативной метрикой;
- при больших значениях модель работает отлично (RMSE и R² это подтверждают), но на малых — «проваливается».

Можем сделать вывод, что модель в целом очень хорошо описывает данные (высокое R²) и средняя абсолютная ошибка в абсолютных значениях умеренная (RMSE). Однако при прогнозировании малых значений модель нестабильна, что отражается в большом MAPE.